In [53]:
import scanpy as sc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
from sklearn.metrics import f1_score

In [54]:
repeat_times = 3
simple_path = '/home/cavin/jt/benchmark/data/hbc/s1_filtered_cells.h5ad'
celltype_col = 'cell_type'
cell_emb_col = 'X_scFoundation'
save_path = f"/home/cavin/jt/benchmark/experiments/embedings/batch_integrate/scfoundation_human_breast_cancer_emb.parquet"


In [55]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [56]:
adata.obsm[cell_emb_col] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial', 'X_scFoundation'

In [57]:
X = adata.obsm[cell_emb_col].astype(np.float32)   # shape: (n_cells, n_dims)
y_raw = adata.obs[celltype_col].values

le = LabelEncoder()
y = le.fit_transform(y_raw)                          # 编码为整数
n_classes = len(le.classes_)
n_features = X.shape[1]

In [58]:
class MLP(nn.Module):
    def __init__(self,input_dim, hidden_dim, n_classes):
        super().__init__()
        self.net = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ReLU(),

                nn.Linear(hidden_dim, hidden_dim),
                nn.LayerNorm(hidden_dim),
                nn.ReLU(),

                nn.Linear(hidden_dim, n_classes)
            )
    def forward(self,x):
        return self.net(x)

In [59]:
# 单折训练函数
def train_one_fold(X_train, y_train, X_val, y_val,
                   input_dim, n_classes,
                   hidden_dim=16, lr=1e-4, epochs=2, batch_size=256,
                   device='cuda'):

    train_ds = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.long)
    )
    val_ds = TensorDataset(
        torch.tensor(X_val,   dtype=torch.float32),
        torch.tensor(y_val,   dtype=torch.long)
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    model = MLP(input_dim, hidden_dim, n_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr,betas=(0.9, 0.999))

    epoch_bar = tqdm(range(epochs), desc=f"  Training", unit="epoch", leave=False)
    for epoch in epoch_bar:
        model.train()
        epoch_loss = 0.0
        n_batches  = 0

    
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            n_batches  += 1

        avg_loss = epoch_loss / n_batches
        epoch_bar.set_postfix(avg_loss=f"{avg_loss:.4f}")

    # 验证
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            preds = model(xb.to(device)).argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(yb.numpy())

    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    acc = (all_preds == all_labels).mean()
    return acc, all_preds, all_labels


In [60]:
# 5 折分层交叉验证
def main(seed:int):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)



    fold_macro_f1s = [] 
    fold_micro_f1s = [] 

    all_val_preds  = np.zeros(len(y), dtype=int)
    all_val_labels = np.zeros(len(y), dtype=int)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"Fold {fold+1}/5")

        X_train, y_train = X[train_idx], y[train_idx]
        X_val,   y_val   = X[val_idx],   y[val_idx]

        
        acc, preds, labels = train_one_fold(
            X_train, y_train, X_val, y_val,
            input_dim=n_features,
            n_classes=n_classes,
            device=device
        )

        macro_f1 = f1_score(labels, preds, average='macro')
        micro_f1 = f1_score(labels, preds, average='micro')

        all_val_preds[val_idx]  = preds
        all_val_labels[val_idx] = labels
        
        fold_macro_f1s.append(macro_f1)
        fold_micro_f1s.append(micro_f1)

        print(f'val_acc={acc:.4f}, macro_f1={macro_f1:.4f}, micro_f1={micro_f1:.4f}')
    return (fold_macro_f1s,fold_micro_f1s)
    

In [61]:
macro_f1 = []
micro_f1 = []
for i in range(repeat_times):
    fold_macro_f1s,fold_micro_f1s = main((i+1) * 2026 )
    macro_f1.extend(fold_macro_f1s)
    micro_f1.extend(fold_micro_f1s)
macro_f1.append(np.mean(macro_f1))
micro_f1.append(np.mean(micro_f1))

Fold 1/5


val_acc=0.7598, macro_f1=0.4010, micro_f1=0.7598
Fold 2/5


val_acc=0.7508, macro_f1=0.3973, micro_f1=0.7508
Fold 3/5


val_acc=0.7589, macro_f1=0.4259, micro_f1=0.7589
Fold 4/5


val_acc=0.7543, macro_f1=0.4002, micro_f1=0.7543
Fold 5/5


...

val_acc=0.7444, macro_f1=0.3782, micro_f1=0.7444
Fold 2/5


val_acc=0.7406, macro_f1=0.3467, micro_f1=0.7406
Fold 3/5


val_acc=0.7541, macro_f1=0.4235, micro_f1=0.7541
Fold 4/5


val_acc=0.7418, macro_f1=0.3691, micro_f1=0.7418
Fold 5/5


val_acc=0.7480, macro_f1=0.3807, micro_f1=0.7480


In [62]:
method = cell_emb_col[2:]

In [63]:
index = [method]
columns = [f"times {i}" for i in range(len(macro_f1) - 1)] + ["mean value"]
macro_df = pd.DataFrame(macro_f1,index=columns,columns=index).T
micro_df = pd.DataFrame(micro_f1,index=columns,columns=index).T

In [64]:
macro_save_path = "/home/cavin/jt/benchmark/experiments/results/cell_type_annotation/repeat_3_times_classfication_macro_metrics.csv"
micro_save_path = "/home/cavin/jt/benchmark/experiments/results/cell_type_annotation/repeat_3_times_classfication_micro_metrics.csv"

In [65]:
macro_df.to_csv(macro_save_path, index=True,mode="a", header=not pd.io.common.file_exists(macro_save_path))
micro_df.to_csv(micro_save_path, index=True,mode="a", header=not pd.io.common.file_exists(micro_save_path))